# Localised LSTM fh=1 modified

This notebook reports `avg_normalised_loss`, `avg_actual_loss`, saves base/reconciled/naive forecasts, and evaluates MAE, MASE, RMSSE.

For LLSTM, each partition/local model applies early stopping independently using the same `args_EARLY_STOP_TOL` and `args_MIN_EPOCHS`. The aggregate loss table is for reporting only.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from llstm import (
    Datacollector,
    run_localised,
    compute_metrics_from_dicts,
    make_reconciliation_frames,
    evaluate_reconciliation_results,
)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [2]:
# -------------------------
# Parameter setup
# -------------------------
#args_path = 'C:/Users/n12553263/yjzPyR/Datasets/GEFCOM2017/GEF2017_univarivate.csv'
args_path = 'E:/yjz/Datasets/GEFCOM2017/Gef2017_univarivate.csv'
args_fh  = 2
args_lags = 48
args_INPUT, args_HIDDEN, args_LAYERS, args_DROP = 1, 64, 2, 0.1
args_EPOCHS, args_BATCH_SIZE, args_LR = 100 + args_fh*50, 256, 1e-3
args_TRAIN_RATIO = 0.8
args_MODEL_NAME = 'llstm'
args_truncated = None
args_DROP_BOUNDARY_GAP = True

# Early stopping settings
args_EARLY_STOP_ENABLED = True
args_EARLY_STOP_TOL = 1e-5
args_MIN_EPOCHS = 20 + args_fh*50

dict_ = Datacollector(pd.read_csv(args_path), lags=args_lags, ts=range(15), fh=args_fh)
df = pd.concat([dict_[key] for key in dict_.keys()]).dropna()
df['ds'] = pd.to_datetime(df['ds'], format='mixed')
if args_truncated is not None:
    df = df[df['ds'] >= args_truncated].reset_index(drop=True)
df = df.reset_index(drop=True)

parts = sorted(df['partition_id'].unique())
lag_cols_reversed = list(reversed(sorted([c for c in df.columns if c.startswith('lags_')], key=lambda x: int(x.split('_')[1]))))
forecast_horizon = ['y'] + sorted([c for c in df.columns if c.startswith('post_')], key=lambda x: int(x.split('_')[1]))
df2 = df[['unique_id', 'partition_id', 'ds'] + lag_cols_reversed + forecast_horizon].copy()

print(df2.head())
print(f'num partitions: {len(parts)}')
print(f'X shape: {df2[lag_cols_reversed].shape} | y shape: {df2[forecast_horizon].shape}')

100%|██████████████████████████████████████████████████████████████████████████████████| 15/15 [00:00<00:00, 18.36it/s]


  unique_id  partition_id                  ds  lags_48  lags_47  lags_46  \
0     Total             0 2003-03-03 00:00:00  12864.0  12389.0  12155.0   
1     Total             0 2003-03-03 01:00:00  12389.0  12155.0  12072.0   
2     Total             0 2003-03-03 02:00:00  12155.0  12072.0  12162.0   
3     Total             0 2003-03-03 03:00:00  12072.0  12162.0  12569.0   
4     Total             0 2003-03-03 04:00:00  12162.0  12569.0  13238.0   

   lags_45  lags_44  lags_43  lags_42  ...   lags_7   lags_6   lags_5  \
0  12072.0  12162.0  12569.0  13238.0  ...  16281.0  16842.0  16503.0   
1  12162.0  12569.0  13238.0  14191.0  ...  16842.0  16503.0  15815.0   
2  12569.0  13238.0  14191.0  15213.0  ...  16503.0  15815.0  14745.0   
3  13238.0  14191.0  15213.0  15646.0  ...  15815.0  14745.0  13444.0   
4  14191.0  15213.0  15646.0  15653.0  ...  14745.0  13444.0  12350.0   

    lags_4   lags_3   lags_2   lags_1        y   post_1   post_2  
0  15815.0  14745.0  13444.0  12350.0

In [3]:
%%time
results = run_localised(
    df=df2,
    lag_cols_reversed=lag_cols_reversed,
    forecast_horizon=forecast_horizon,
    input_size=args_INPUT,
    hidden_size=args_HIDDEN,
    num_layers=args_LAYERS,
    dropout=args_DROP,
    batch_size=args_BATCH_SIZE,
    epochs=args_EPOCHS,
    lr=args_LR,
    train_ratio=args_TRAIN_RATIO,
    partition_col='partition_id',
    device=None,
    clip_grad=1.0,
    verbose=True,
    lags=args_lags,
    fh=args_fh,
    time_col='ds',
    drop_boundary_gap=args_DROP_BOUNDARY_GAP,
    early_stop_enabled=args_EARLY_STOP_ENABLED,
    early_stop_tol=args_EARLY_STOP_TOL,
    min_epochs=args_MIN_EPOCHS,
)

parts = results['parts']
train_idx = results['train_idx']
test_idx = results['test_idx']
dict_pred = results['dict_pred']
dict_true = results['dict_true']
dict_tr_pred = results['dict_tr_pred']
dict_tr_true = results['dict_tr_true']
dict_naive = results['dict_naive']

print('Device:', results['device'])
print('Stopped early:', results['stopped_early'])
print('Final avg_normalised_loss:', results['final_avg_normalised_loss'])
print('Final avg_actual_loss:', results['final_avg_actual_loss'])
print('Early stop rule: each local model checks its own avg_normalised_loss delta independently.')
local_stop_summary_df = pd.DataFrame(results['local_stop_summary'])
display(local_stop_summary_df.tail())
pd.DataFrame(results['round_logs']).tail()

[Localised] pid=0 epoch 001/200 | avg_normalised_loss=0.033948 | avg_actual_loss=284595.992469 | delta=nan | stopped=False
[Localised] pid=0 epoch 002/200 | avg_normalised_loss=0.020643 | avg_actual_loss=173053.713013 | delta=0.0133054 | stopped=False
[Localised] pid=0 epoch 003/200 | avg_normalised_loss=0.015670 | avg_actual_loss=131368.560971 | delta=0.00497244 | stopped=False
[Localised] pid=0 epoch 004/200 | avg_normalised_loss=0.012286 | avg_actual_loss=102994.182911 | delta=0.00338466 | stopped=False
[Localised] pid=0 epoch 005/200 | avg_normalised_loss=0.010661 | avg_actual_loss=89377.510779 | delta=0.00162427 | stopped=False
[Localised] pid=0 epoch 006/200 | avg_normalised_loss=0.009690 | avg_actual_loss=81232.134183 | delta=0.000971627 | stopped=False
[Localised] pid=0 epoch 007/200 | avg_normalised_loss=0.007961 | avg_actual_loss=66739.466294 | delta=0.00172877 | stopped=False
[Localised] pid=0 epoch 008/200 | avg_normalised_loss=0.008623 | avg_actual_loss=72292.274253 | delt

,partition_id,stopped_early,stop_epoch,stop_reason,early_stop_tol,min_epochs,final_avg_normalised_loss,final_avg_actual_loss,n_train,n_test
5,5,True,134,normalised_loss_delta<1e-05 after min_epochs=120,0.00001,120,0.004840,203.772932,99286,24833
6,6,True,150,normalised_loss_delta<1e-05 after min_epochs=120,0.00001,120,0.005139,67.978491,99286,24833
7,7,True,127,normalised_loss_delta<1e-05 after min_epochs=120,0.00001,120,0.004069,1311.751157,99286,24833
8,8,True,122,normalised_loss_delta<1e-05 after min_epochs=120,0.00001,120,0.004073,620.021531,99286,24833
9,9,True,150,normalised_loss_delta<1e-05 after min_epochs=120,0.00001,120,0.003563,554.608756,99286,24833


CPU times: total: 4h 25min 5s
Wall time: 1h 48min 33s


,epoch,avg_normalised_loss,avg_actual_loss,normalised_loss_delta,n_active_partitions_reported,n_partitions_stopped_this_epoch
180,181,0.007150,327.192074,0.000062,1,0
181,182,0.007789,356.413872,0.000639,1,0
182,183,0.006975,319.173335,0.000814,1,0
183,184,0.007001,320.346718,0.000026,1,0
184,185,0.006992,319.947963,0.000009,1,1


In [4]:
# Base model and Naive baseline metrics before reconciliation
base_metrics = compute_metrics_from_dicts(dict_true, dict_pred, dict_train_true=dict_tr_true, h_idx=0)
naive_metrics = compute_metrics_from_dicts(dict_true, dict_naive, dict_train_true=dict_tr_true, h_idx=0)
display(base_metrics.tail())
display(naive_metrics.tail())

,partition_id,n_test,MAE,MASE,RMSSE
6,6,24833,8.629735,0.349986,0.373002
7,7,24833,19.440744,0.182202,0.185056
8,8,24833,14.926017,0.193332,0.197760
9,9,24833,14.441606,0.185404,0.192769
10,Overall,248330,23.159314,0.193477,0.199309


,partition_id,n_test,MAE,MASE,RMSSE
6,6,24833,23.279350,0.944114,0.890544
7,7,24833,99.784432,0.935197,0.924278
8,8,24833,70.843988,0.917619,0.888004
9,9,24833,70.182203,0.901010,0.876564
10,Overall,248330,130.504095,0.920061,0.898401


In [5]:
from hierarchicalforecast.core import HierarchicalReconciliation
from hierarchicalforecast.methods import BottomUp, MinTrace

#path = 'C:/Users/n12553263/yjzPyR/Datasets/GEFCOM2017/'
path = 'E:/yjz/Datasets/GEFCOM2017/'
S = pd.read_csv(f'{path}GEF_S.csv')
tags = pd.read_pickle(f'{path}tags.pkl')
def align_S_for_hierarchicalforecast(S, id_col="unique_id"):
    S = S.copy()
    if id_col not in S.columns:
        S = S.reset_index()
        if id_col not in S.columns:
            S = S.rename(columns={S.columns[0]: id_col})
    bottom_ids = [c for c in S.columns if c != id_col]
    agg_ids = [uid for uid in S[id_col].tolist() if uid not in bottom_ids]
    row_order = agg_ids + bottom_ids
    S_aligned = (
        S
        .set_index(id_col)
        .loc[row_order, bottom_ids]
        .reset_index())
    bottom_block = S_aligned[bottom_ids].tail(len(bottom_ids)).to_numpy()
    if not np.allclose(bottom_block, np.eye(len(bottom_ids))):
        raise ValueError("S Error")
    return S_aligned

S = align_S_for_hierarchicalforecast(S)
reconciliation_methods = [
    BottomUp(),
    MinTrace(method='mint_shrink'),
    MinTrace(method='wls_var'),
    MinTrace(method='ols'),
    MinTrace(method='wls_struct'),
]
reconcilers = HierarchicalReconciliation(reconcilers=reconciliation_methods)

recon_results = {}
train_frames = {}
test_frames = {}
for H_IDX in range(args_fh + 1):
    y_df, y_hat_df, train_frame, test_frame = make_reconciliation_frames(
        df=df2,
        train_idx=train_idx,
        test_idx=test_idx,
        parts=parts,
        dict_tr_pred=dict_tr_pred,
        dict_pred=dict_pred,
        forecast_horizon=forecast_horizon,
        base_model_name=args_MODEL_NAME,
        horizon_idx=H_IDX,
    )
    train_frames[H_IDX] = train_frame
    test_frames[H_IDX] = test_frame
    recon_results[H_IDX] = reconcilers.reconcile(Y_hat_df=y_hat_df, S=S, tags=tags, Y_df=y_df)

recon_results[0].head()

C:\Users\qifei\AppData\Local\Temp\ipykernel_7164\2550965230.py:54: DeprecationWarning: The 'S' parameter is deprecated and will be removed in a future version. Please use 'S_df' instead.
  recon_results[H_IDX] = reconcilers.reconcile(Y_hat_df=y_hat_df, S=S, tags=tags, Y_df=y_df)
C:\Users\qifei\AppData\Local\Temp\ipykernel_7164\2550965230.py:54: DeprecationWarning: The 'S' parameter is deprecated and will be removed in a future version. Please use 'S_df' instead.
  recon_results[H_IDX] = reconcilers.reconcile(Y_hat_df=y_hat_df, S=S, tags=tags, Y_df=y_df)
C:\Users\qifei\AppData\Local\Temp\ipykernel_7164\2550965230.py:54: DeprecationWarning: The 'S' parameter is deprecated and will be removed in a future version. Please use 'S_df' instead.
  recon_results[H_IDX] = reconcilers.reconcile(Y_hat_df=y_hat_df, S=S, tags=tags, Y_df=y_df)


,unique_id,ds,y,llstm,llstm/BottomUp,llstm/MinTrace_method-mint_shrink,llstm/MinTrace_method-wls_var,llstm/MinTrace_method-ols,llstm/MinTrace_method-wls_struct
0,Total,2014-06-30 23:00:00,15138.0,15099.453125,15036.901733,15036.868788,15037.829419,15085.762939,15053.959995
1,Total,2014-07-01 00:00:00,13810.0,13803.911133,13740.858337,13742.233670,13743.384622,13790.760367,13759.665039
2,Total,2014-07-01 01:00:00,12979.0,12936.737305,12922.407593,12917.658389,12917.263693,12931.415762,12920.889850
3,Total,2014-07-01 02:00:00,12468.0,12392.079102,12464.383850,12445.831625,12442.310942,12399.336177,12423.394093
4,Total,2014-07-01 03:00:00,12226.0,12147.507812,12212.263000,12193.929108,12190.553190,12153.215025,12173.586443


In [6]:
# Save base, reconciled and naive forecasts; evaluate MAE, MASE and RMSSE
output_prefix = f'fh{args_fh+1}_loc'
eval_outputs = evaluate_reconciliation_results(
    recon_results=recon_results,
    train_frames=train_frames,
    test_frames=test_frames,
    forecast_horizon=forecast_horizon,
    base_model_name=args_MODEL_NAME,
    output_prefix=output_prefix,
    approach='localised',
    round_logs=results['round_logs'],
    timings=results['timings'],
)

forecast_table = eval_outputs['forecast_table']
per_series_metrics = eval_outputs['per_series_metrics']
metrics_by_level = eval_outputs['metrics_by_level']
overall_metrics = eval_outputs['overall_metrics']
timing_df = eval_outputs['timing_df']
output_paths = eval_outputs['output_paths']

# LLSTM-specific early stopping diagnostics: each local model has its own stop epoch.
local_stop_summary_path = f'{output_prefix}_local_stop_summary.csv'
round_logs_by_partition_path = f'{output_prefix}_round_logs_by_partition.csv'
pd.DataFrame(results['local_stop_summary']).to_csv(local_stop_summary_path, index=False)
pd.DataFrame(results['round_logs_by_partition']).to_csv(round_logs_by_partition_path, index=False)
output_paths['local_stop_summary'] = local_stop_summary_path
output_paths['round_logs_by_partition'] = round_logs_by_partition_path

print('Saved outputs:')
for k, v in output_paths.items():
    print(f'  {k}: {v}')

metrics_by_level

Saved outputs:
  forecasts: fh3_loc_forecasts.csv
  per_series_metrics: fh3_loc_per_series_metrics.csv
  metrics_by_level: fh3_loc_metrics_by_level.csv
  overall_metrics: fh3_loc_overall_metrics.csv
  round_logs: fh3_loc_round_logs.csv
  timing: fh3_loc_timing.csv
  local_stop_summary: fh3_loc_local_stop_summary.csv
  round_logs_by_partition: fh3_loc_round_logs_by_partition.csv


,level,role,method,MAE,MASE,RMSSE,n_series,approach
0,1,top,base,142.799619,0.246840,0.252945,1,localised
1,1,top,bu,131.787363,0.227804,0.236582,1,localised
2,1,top,mint_ols,137.884562,0.238344,0.245231,1,localised
3,1,top,mint_shrinkage,129.276520,0.223464,0.232528,1,localised
4,1,top,mint_var,128.821463,0.222678,0.231760,1,localised
5,1,top,naive,1015.797221,1.755882,1.687738,1,localised
6,1,top,wls_structure,129.540923,0.223921,0.232498,1,localised
7,2,middle,base,29.412201,0.341151,0.357559,6,localised
8,2,middle,bu,28.865955,0.339051,0.355154,6,localised
9,2,middle,mint_ols,31.202542,0.403188,0.407541,6,localised


In [7]:
overall_metrics

,method,MAE,MASE,RMSSE,n_series,approach
0,base,40.484899,0.327879,0.342575,10,localised
1,bu,39.055925,0.324716,0.339496,10,localised
2,mint_ols,41.087855,0.364795,0.371975,10,localised
3,mint_shrinkage,38.388984,0.320940,0.335899,10,localised
4,mint_var,38.316581,0.320810,0.335674,10,localised
5,naive,250.744261,1.760754,1.686418,10,localised
6,wls_structure,38.685189,0.327788,0.339823,10,localised


In [8]:
timing_df

,module,seconds
0,training_sec,6512.515958
1,total_sec,6513.605645
